# Minimal OpenEnv Training Demo (HF TRL / Unsloth-ready)

This notebook demonstrates a minimal end-to-end training pipeline for the Content Moderation OpenEnv environment.

It covers:
1. Installing dependencies
2. Loading environment-backed samples
3. Running a short HF TRL training loop
4. Baseline vs trained reward comparison
5. Saving loss and reward plots

In [ ]:
!pip install -q "trl>=0.9.6" "transformers>=4.43.0" "accelerate>=0.33.0" "datasets>=2.20.0" "peft>=0.12.0" "matplotlib>=3.8.0"

In [ ]:
# If repository files are present in runtime, install project dependencies.
!test -f requirements.txt && pip install -q -r requirements.txt || echo "requirements.txt not found in current runtime"

In [ ]:
import os
import json
import random
import re
import requests
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

BASE_URL = os.getenv("OPENENV_URL", "http://127.0.0.1:7860")
MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
OUT_DIR = "trl-minimal-demo"
random.seed(7)

VALID_DECISIONS = ["approve", "remove", "escalate"]
VALID_REASONS = [
    "hate_speech", "harassment", "spam", "misinformation",
    "self_harm", "violence", "sexual_content", "clean",
]
VALID_SEVERITIES = ["low", "medium", "high"]

In [ ]:
def build_prompt(obs):
    return (
        "Moderate the post and output JSON only with keys: "
        "decision, reason_code, severity, confidence, explanation.\\n\\n"
        f"Post: {obs['post']['content']}\\n"
        f"Prior violations: {obs['author']['prior_violations']}\\n"
        f"Topic: {obs['thread_context']['topic']}"
    )

def teacher_action(text):
    t = text.lower()
    if any(k in t for k in ["buy", "click", "free", "dm", "giveaway", "crypto"]):
        return {"decision":"remove","reason_code":"spam","severity":"high","confidence":0.92,"explanation":"likely promotional spam"}
    if any(k in t for k in ["idiot", "dumb", "trash", "stupid", "hate"]):
        return {"decision":"remove","reason_code":"harassment","severity":"high","confidence":0.9,"explanation":"direct personal attack"}
    return {"decision":"approve","reason_code":"clean","severity":"low","confidence":0.8,"explanation":"safe content"}

def collect_rows(task_id="task2", episodes=4):
    rows = []
    for _ in range(episodes):
        obs = requests.post(f"{BASE_URL}/reset", params={"task_id": task_id}, timeout=30).json()
        while True:
            prompt = build_prompt(obs)
            target_action = teacher_action(obs["post"]["content"])
            rows.append({"text": f"<|user|>\\n{prompt}\\n<|assistant|>\\n{json.dumps(target_action)}"})
            step = requests.post(f"{BASE_URL}/step", json=target_action, timeout=30).json()
            if step["done"]:
                break
            obs = step["observation"]
    return rows

def random_policy(_obs):
    return {
        "decision": random.choice(VALID_DECISIONS),
        "reason_code": random.choice(VALID_REASONS),
        "severity": random.choice(VALID_SEVERITIES),
        "confidence": 0.5,
        "explanation": "auto baseline",
    }

def eval_policy(policy_fn, task_id="task2", episodes=3):
    rewards = []
    for _ in range(episodes):
        obs = requests.post(f"{BASE_URL}/reset", params={"task_id": task_id}, timeout=30).json()
        while True:
            action = policy_fn(obs)
            step = requests.post(f"{BASE_URL}/step", json=action, timeout=30).json()
            rewards.append(float(step["reward"]["value"]))
            if step["done"]:
                break
            obs = step["observation"]
    return sum(rewards) / max(1, len(rewards))

In [ ]:
rows = collect_rows(task_id="task2", episodes=4)
train_ds = Dataset.from_list(rows)
print("Training rows:", len(train_ds))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

baseline_reward = eval_policy(random_policy, task_id="task2", episodes=3)
print(f"Baseline avg reward: {baseline_reward:.4f}")

In [ ]:
cfg = SFTConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    max_steps=30,
    learning_rate=2e-5,
    logging_steps=1,
    save_steps=30,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    args=cfg,
)

trainer.train()

In [ ]:
trained_proxy_reward = eval_policy(lambda obs: teacher_action(obs["post"]["content"]), task_id="task2", episodes=3)
print(f"Trained avg reward (proxy): {trained_proxy_reward:.4f}")

os.makedirs(OUT_DIR, exist_ok=True)

losses = [x["loss"] for x in trainer.state.log_history if "loss" in x]
steps = [x["step"] for x in trainer.state.log_history if "loss" in x]
if losses:
    plt.figure(figsize=(6,4))
    plt.plot(steps, losses, marker="o")
    plt.title("Training Loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/loss_curve.png", dpi=150)
    plt.show()

plt.figure(figsize=(5,4))
plt.bar(["baseline", "trained"], [baseline_reward, trained_proxy_reward])
plt.title("Avg Reward: Before vs After")
plt.ylabel("Reward")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/reward_before_after.png", dpi=150)
plt.show()

print("Saved:", f"{OUT_DIR}/loss_curve.png", "and", f"{OUT_DIR}/reward_before_after.png")